In [17]:
import os
import random
import pandas as pd
import numpy as np
import pyemu

In [3]:
# set up path
target_dir = r"C:\Python\Personal\proj6\codes\sandbox"
rel_t_d = os.path.relpath(target_dir, os.getcwd()) 
os.chdir(rel_t_d)

In [40]:
################
### well_pos ###
################

# specify number of wells
n_wells = 2

# specify bounds of coordinate system
x_min, x_max = 0.1, 100
y_min, y_max = 0.1, 100

# build parameter data frame
par_names = []
for i in range(1, n_wells + 1):
    par_names.extend([f"well_{i}_x", f"well_{i}_y"])

# initialize an empty PEST control file (v2)
pst = pyemu.pst_utils.generic_pst(par_names=par_names)

# populate PEST control file (v2) metadata
pdf = pst.parameter_data
pdf.loc[:, "partrans"] = "none"
pdf.loc[:, "parchglim"] = "factor"
pdf.loc[:, "parlbnd"] = x_min
pdf.loc[:, "parubnd"] = x_max
pdf.loc[:, "pargp"] = "well_pos"

pdf.loc[:, "parval1"] = np.random.uniform(x_min, x_max, size=len(pdf))

# specify how may individuals in the population in the decision variable space
num_reals = 150

# generate initial population using PESTPP-MOU
wellpop = pyemu.ParameterEnsemble.from_uniform_draw(pst, num_reals = num_reals)

# select a random realization from the initial population
# pick one index name at random from the ensemble
selected_real = np.random.choice(wellpop.index)
print(f"QC: Selecting realization '{selected_real}' to populate parval1 in the PST file.")
pst.parameter_data.loc[par_names, "parval1"] = wellpop.loc[selected_real, par_names].values

# set control data for PEST control file (v2)
pst.control_data.pestmode = "estimation"
pst.control_data.noptmax = 0

# write PEST control file (v2)
pst_filename = "fwd_model.pst"
pst.write(pst_filename, version = 2)

# record to external file in the current directory
wellpop.to_csv("initial_wellpop.csv")

QC: Selecting realization '86' to populate parval1 in the PST file.
noptmax:0, npar_adj:4, nnz_obs:1


In [41]:
##################################################
### two well example visualization of well_pos ###
##################################################

import plotly.graph_objects as go

fig = go.Figure()

# access the underlying dataframe using the internal _df attribute
df = wellpop._df

# set styles

# well 1 ensemble
fig.add_trace(go.Scatter(
    x=df["well_1_x"], y=df["well_1_y"],
    mode='markers', 
    marker=dict(color='lightskyblue', opacity=0.3, symbol='circle'),
    name="Well 1 Ensemble"
))

# well 1 selected
fig.add_trace(go.Scatter(
    x=[pst.parameter_data.loc["well_1_x", "parval1"]], 
    y=[pst.parameter_data.loc["well_1_y", "parval1"]],
    mode='markers+text', text=["Well 1"], textposition="top center",
    marker=dict(color='dodgerblue', size=10, symbol='circle', 
                line=dict(width=1, color='dodgerblue')),
    name="Well 1 Selected (parval1)"
))

# well 2 ensemble
fig.add_trace(go.Scatter(
    x=df["well_2_x"], y=df["well_2_y"],
    mode='markers', 
    marker=dict(color='lightcoral', opacity=0.3, symbol='square'),
    name="Well 2 Ensemble"
))

# well 2 selected
fig.add_trace(go.Scatter(
    x=[pst.parameter_data.loc["well_2_x", "parval1"]], 
    y=[pst.parameter_data.loc["well_2_y", "parval1"]],
    mode='markers+text', text=["Well 2"], textposition="top center",
    marker=dict(color='crimson', size=10, symbol='square', 
                line=dict(width=1, color='crimson')),
    name="Well 2 Selected (parval1)"
))

# add boundary box
fig.add_shape(type="rect", x0=x_min, y0=y_min, x1=x_max, y1=y_max, 
              line=dict(color="black", dash="dash"))

fig.update_layout(
    xaxis=dict(range=[-5, 105], title="X Coordinate"),
    yaxis=dict(range=[-5, 105], title="Y Coordinate"),
    title=f"Well Locations (Realization: {selected_real})",
    template="plotly_white",
    width=700, height=700
)

fig.show()
